# Region Meshes — Grids & Hexbins

Unified replacement for `00d_exploratory_region_hexbins` and `00e_exploratory_region_grids`.
Builds square-grid and/or hexagonal meshes over Ugandan refugee-hosting regions,
attributes each cell with region and settlement metadata, and exports to GeoJSON.

| Section | Contents |
|---|---|
| 1 | Parameters, imports, paths, data loading |
| 2 | Exploratory overview plots |
| 3 | Mesh generation functions |
| 4 | Build, attribute, and export meshes |

## Section 1 — Setup

In [ ]:
# ============================================================
# PARAMETERS — edit only this cell
# ============================================================

# Geometry types to build.  Include one, both, or neither.
GEOMETRY_TYPES = ["grids", "hexbins"]

# Cell sizes to build, as {label: area_m2}.
# 250k ≈ 500×500 m  |  62k ≈ 250×250 m  |  15k ≈ 125×125 m
AREAS = {
    "250k": 250_000,
    "62k":   62_500,
    "15k":   15_625,
}

# Buffer around region boundaries used to retain cells (meters)
BUFFER_DIST = 10_000

# Projected CRS for all spatial operations
PROJECTED_CRS = "EPSG:32636"

# Write output GeoJSONs to data/processed
EXPORT_RESULTS = True

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
from math import sqrt
from pathlib import Path
from shapely.geometry import box, Point, Polygon

In [ ]:
def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "paths.yaml").exists():
            return candidate
    raise FileNotFoundError("configs/paths.yaml not found")

project_root = find_project_root()

with open(project_root / "configs" / "paths.yaml") as f:
    paths = yaml.safe_load(f)

data_dir      = project_root / paths["data"]["processed"]
data_external = project_root / paths["data"]["external"]
output_dir    = data_dir
maps_dir      = project_root / paths["outputs"]["dynamic_maps"]

In [ ]:
settlements = gpd.read_file(
    data_dir / "UNHCR_poc_boundaries-Uganda_attributed_deduped.geojson"
).to_crs(PROJECTED_CRS)

refugee_regions = gpd.read_file(data_external / "refugeehosting_regions.geojson")
refugee_regions = gpd.GeoDataFrame(
    refugee_regions[["ADM2_EN", "ADM1_EN", "ADM0_EN", "geometry"]],
    geometry="geometry",
).to_crs(PROJECTED_CRS)

print(f"Settlements: {len(settlements):,}")
print(f"Refugee regions: {len(refugee_regions):,}")

## Section 2 — Exploratory Overview

In [ ]:
# Region and settlement overview with Nakivale / Bidi Bidi callout boxes
nakivale  = settlements[settlements["name"].str.contains("nakiv", case=False, na=False)]
bidi_bidi = settlements[settlements["name"].str.contains("bidi",  case=False, na=False)]

def bounding_box(gdf, buffer=0):
    minx, miny, maxx, maxy = gdf.buffer(buffer).total_bounds
    return gpd.GeoDataFrame(geometry=[box(minx, miny, maxx, maxy)], crs=gdf.crs)

fig, ax = plt.subplots(figsize=(8, 10))

refugee_regions.plot(ax=ax, color="#c7e9c0", edgecolor="black", alpha=0.35, linewidth=0.8)
settlements.boundary.plot(ax=ax, edgecolor="#8c2d04", alpha=0.9, linewidth=1.4, label="Settlements")
bounding_box(nakivale,  buffer=2000).boundary.plot(ax=ax, edgecolor="#2b8cbe", linewidth=2, linestyle="--", label="Nakivale")
bounding_box(bidi_bidi, buffer=0   ).boundary.plot(ax=ax, edgecolor="#fdae6b", linewidth=2, linestyle="--", label="Bidi Bidi")

ax.set_title("Refugee Settlement Containing Regions", fontsize=15, pad=10)
ax.set_xticks([]); ax.set_yticks([])
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# Zoom plots for two focal settlements
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, sett, color, title in [
    (axes[0], nakivale,  "#2b8cbe", "Nakivale"),
    (axes[1], bidi_bidi, "#fdae6b", "Bidi Bidi"),
]:
    refugee_regions.plot(ax=ax, color="#f5f5f5", edgecolor="none")
    sett.plot(ax=ax, color=color, edgecolor="black", alpha=0.6, linewidth=1)
    minx, miny, maxx, maxy = sett.buffer(1000).total_bounds
    ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
    ax.set_title(f"{title} Refugee Settlement", fontsize=14, pad=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## Section 3 — Mesh Generation Functions

In [ ]:
def make_grid_mesh_over_union(polys_gdf, cell_area, buffer_dist=10_000, origin=None):
    """
    Build a square grid covering a polygon region, keeping only cells whose
    centroids fall within `buffer_dist` meters of the input polygons.

    Parameters
    ----------
    polys_gdf  : GeoDataFrame  — input polygons in a projected CRS (meters)
    cell_area  : float         — square cell area in square meters
    buffer_dist: float         — retention buffer in meters (default 10 km)
    origin     : (x0, y0)      — optional grid origin for alignment across runs

    Returns
    -------
    GeoDataFrame of grid polygons
    """
    if polys_gdf.crs is None or polys_gdf.crs.is_geographic:
        raise ValueError("Input polygons must be in a projected CRS (meters).")

    side = sqrt(cell_area)

    buffered       = polys_gdf.buffer(buffer_dist)
    buffered_union = buffered.union_all()
    minx, miny, maxx, maxy = buffered.total_bounds

    x0, y0 = (minx - side, miny - side) if origin is None else origin

    xs = np.arange(x0, maxx + side, side)
    ys = np.arange(y0, maxy + side, side)

    cells = [
        box(x, y, x + side, y + side)
        for x in xs
        for y in ys
        if buffered_union.contains(Point(x + side / 2, y + side / 2))
    ]

    return gpd.GeoDataFrame(geometry=cells, crs=polys_gdf.crs)

In [ ]:
def make_hex_mesh_over_union(polys_gdf, hex_area, buffer_dist=10_000, origin=None):
    """
    Build a hexagonal grid covering a polygon region, keeping only cells whose
    centroids fall within `buffer_dist` meters of the input polygons.

    Parameters
    ----------
    polys_gdf  : GeoDataFrame  — input polygons in a projected CRS (meters)
    hex_area   : float         — target hexagon area in square meters
    buffer_dist: float         — retention buffer in meters (default 10 km)
    origin     : (x0, y0)      — optional grid origin for alignment across runs

    Returns
    -------
    GeoDataFrame of hexagon polygons
    """
    if polys_gdf.crs is None or polys_gdf.crs.is_geographic:
        raise ValueError("Input polygons must be in a projected CRS (meters).")

    s  = sqrt((2 * hex_area) / (3 * sqrt(3)))   # side length
    w  = 2 * s                                   # flat-to-flat width
    h  = sqrt(3) * s                             # height
    dx = 1.5 * s                                 # column x-step
    dy = h                                       # row y-step

    buffered       = polys_gdf.buffer(buffer_dist)
    buffered_union = buffered.union_all()
    minx, miny, maxx, maxy = buffered.total_bounds

    x0, y0 = (minx - w, miny - h) if origin is None else origin

    cols = np.arange(x0, maxx + w, dx)
    rows = np.arange(y0, maxy + h, dy)

    hexes = []
    for i, x in enumerate(cols):
        for j, y in enumerate(rows):
            y_off = y + (dy / 2 if i % 2 else 0)
            if buffered_union.contains(Point(x, y_off)):
                vertices = [
                    (x + s * np.cos(np.deg2rad(a)),
                     y_off + s * np.sin(np.deg2rad(a)))
                    for a in range(0, 360, 60)
                ]
                hexes.append(Polygon(vertices))

    return gpd.GeoDataFrame(geometry=hexes, crs=polys_gdf.crs)

In [ ]:
def process_mesh(mesh_gdf, refugee_regions, settlements, out_path, settlement_col="name"):
    """
    Attribute a raw mesh with region + settlement metadata, compute lat/lon,
    assign sequential OIDs, and optionally save to GeoJSON.

    Parameters
    ----------
    mesh_gdf       : GeoDataFrame — raw mesh (grids or hexbins)
    refugee_regions: GeoDataFrame — region polygons with ADM columns
    settlements    : GeoDataFrame — settlement polygons with 'name' column
    out_path       : Path or None — save destination; None = skip export
    settlement_col : str          — output column name for settlement
                                    ('name' for grids, 'settlement_name' for hexbins)

    Returns
    -------
    Attributed GeoDataFrame
    """
    mesh = mesh_gdf.copy().reset_index(drop=True)
    mesh["OID"] = np.arange(len(mesh))

    # --- Step 1: attribute region (centroid-within-region join) ---
    centroids = mesh.copy()
    centroids["geometry"] = centroids.geometry.centroid

    mesh = (
        gpd.sjoin(centroids, refugee_regions, how="left", predicate="within")
        .merge(mesh[["OID", "geometry"]], on="OID")
        .drop(columns=["index_right"], errors="ignore")
    )

    # --- Step 2: attribute settlement (polygon-intersects-settlement join) ---
    mesh = (
        gpd.sjoin(
            mesh,
            settlements[["name", "geometry"]],
            how="left",
            predicate="intersects",
        )
        .drop(columns=["index_right"], errors="ignore")
    )

    # If a cell straddles multiple settlements, keep the first match
    mesh = mesh.drop_duplicates(subset="OID", keep="first").reset_index(drop=True)
    mesh["OID"] = np.arange(len(mesh))   # re-sequence after dedup

    # Rename settlement name column if required
    if settlement_col != "name" and "name" in mesh.columns:
        mesh = mesh.rename(columns={"name": settlement_col})

    # --- Step 3: lat/lon from projected centroids (avoids geographic CRS warning) ---
    centroids_proj = gpd.GeoSeries(mesh.geometry.centroid, crs=mesh.crs)
    centroids_wgs  = centroids_proj.to_crs(4326)
    mesh["lat"] = centroids_wgs.y
    mesh["lon"] = centroids_wgs.x

    # --- Step 4: canonical column order ---
    id_cols  = ["OID", "ADM2_EN", "ADM1_EN", "ADM0_EN", settlement_col, "lat", "lon"]
    id_cols  = [c for c in id_cols if c in mesh.columns]
    rest     = [c for c in mesh.columns if c not in id_cols and c != "geometry"]
    mesh     = mesh[id_cols + rest + ["geometry"]]

    # --- Step 5: export ---
    if out_path is not None:
        mesh.to_file(out_path, driver="GeoJSON")
        print(f"  Saved {out_path.name}  ({len(mesh):,} cells)")

    return mesh

## Section 4 — Build, Attribute & Export

Runs for every combination of `GEOMETRY_TYPES × AREAS`.
The two cells below are independent: re-run only **Process & export** if the raw meshes
are already built (e.g. to re-run with a changed `EXPORT_RESULTS` flag).

In [ ]:
# Per-type config: builder function, settlement column name, output filename template
MESH_CONFIG = {
    "grids": {
        "builder":        make_grid_mesh_over_union,
        "settlement_col": "name",
        "out_fname":      "uganda_grids_{label}_lcluc_10k_buffer_v3.geojson",
    },
    "hexbins": {
        "builder":        make_hex_mesh_over_union,
        "settlement_col": "settlement_name",
        "out_fname":      "uganda_hexbins_{label}_lcluc_10K_buffer_v2.geojson",
    },
}

In [ ]:
# Build raw meshes
raw_meshes = {}   # {(geom_type, label): GeoDataFrame}

for geom_type in GEOMETRY_TYPES:
    cfg     = MESH_CONFIG[geom_type]
    builder = cfg["builder"]

    for label, area in AREAS.items():
        print(f"Building {geom_type} {label}  ({area:,} m²) ...")
        raw = builder(refugee_regions, area, buffer_dist=BUFFER_DIST)
        raw_meshes[(geom_type, label)] = raw
        print(f"  {len(raw):,} cells")

In [ ]:
# Attribute and export
results = {}   # {(geom_type, label): attributed GeoDataFrame}

for (geom_type, label), raw in raw_meshes.items():
    cfg      = MESH_CONFIG[geom_type]
    out_path = (output_dir / cfg["out_fname"].format(label=label)) if EXPORT_RESULTS else None

    print(f"\nAttributing {geom_type} {label} ...")
    results[(geom_type, label)] = process_mesh(
        raw,
        refugee_regions,
        settlements,
        out_path,
        settlement_col=cfg["settlement_col"],
    )

print("\nDone.")

In [ ]:
# Quick coverage check — cell counts and settlement attribution rate
print(f"{'type':<10} {'label':<6} {'cells':>8}  {'with settlement':>16}")
print("-" * 46)
for (geom_type, label), gdf in results.items():
    cfg     = MESH_CONFIG[geom_type]
    scol    = cfg["settlement_col"]
    n_total = len(gdf)
    n_sett  = gdf[scol].notna().sum() if scol in gdf.columns else 0
    print(f"{geom_type:<10} {label:<6} {n_total:>8,}  {n_sett:>12,} ({100*n_sett/n_total:.1f}%)")

In [ ]:
# Optional: preview a mesh over Nakivale for a chosen type and resolution
PREVIEW_TYPE  = "grids"    # "grids" or "hexbins"
PREVIEW_LABEL = "250k"

key = (PREVIEW_TYPE, PREVIEW_LABEL)
if key in results:
    gdf  = results[key]
    nak  = settlements[settlements["name"].str.contains("nakiv", case=False, na=False)]

    fig, ax = plt.subplots(figsize=(8, 8))
    refugee_regions.plot(ax=ax, color="lightgreen", edgecolor="black", linewidth=0.7)
    gdf.plot(ax=ax, color="white", edgecolor="black", alpha=0.5, linewidth=0.25)
    nak.boundary.plot(ax=ax, edgecolor="red", linewidth=1.5)

    minx, miny, maxx, maxy = nak.buffer(2000).total_bounds
    ax.set_xlim(minx, maxx); ax.set_ylim(miny, maxy)
    ax.set_title(f"{PREVIEW_TYPE} {PREVIEW_LABEL} — Nakivale", fontsize=13)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print(f"{key} not in results — check GEOMETRY_TYPES and AREAS.")